In [1]:
import sys
sys.path.append('..')

from src.embeddings import carregar_nilc

vetores, vocab = carregar_nilc()
print('Matriz de vetores:', vetores.shape, vetores.dtype)
print('Tamanho do vocabulario:', len(vocab))

c:\dev\tech-challenge-fase4\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\henri\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


embeddings.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

embeddings.safetensors: downloading bytes:           |  0.00B            

c:\dev\tech-challenge-fase4\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\henri\.cache\huggingface\hub\models--nilc-nlp--fasttext-skip-gram-300d. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/9.21M [00:00<?, ?B/s]

Matriz de vetores: (929605, 300) float32
Tamanho do vocabulario: 929605


In [2]:
# testando 
testes = ["entrega", "Entrega", "ENTREGA", "atraso", "recomendo", "não", "nao",
          "péssimo", "pessimo", "ótimo", "otimo", "entregue", "entrgue", "recomendooo", "vc"]

for p in testes:
    print(f"{p!r:15} -> {'no vocabulário' if p in vocab else 'FORA do vocabulário'}")

'entrega'       -> no vocabulário
'Entrega'       -> FORA do vocabulário
'ENTREGA'       -> FORA do vocabulário
'atraso'        -> no vocabulário
'recomendo'     -> no vocabulário
'não'           -> no vocabulário
'nao'           -> no vocabulário
'péssimo'       -> no vocabulário
'pessimo'       -> no vocabulário
'ótimo'         -> no vocabulário
'otimo'         -> no vocabulário
'entregue'      -> no vocabulário
'entrgue'       -> FORA do vocabulário
'recomendooo'   -> FORA do vocabulário
'vc'            -> no vocabulário


In [3]:
import numpy as np

palavras = list(vocab.keys())

def vizinhos(palavra, k=8):
    v = vetores[vocab[palavra]]
    normas = np.linalg.norm(vetores, axis=1) * np.linalg.norm(v) + 1e-9
    sims = (vetores @ v) / normas
    melhores = np.argsort(-sims)[1 : k + 1]
    return [(palavras[i], round(float(sims[i]), 3)) for i in melhores]

for p in ["entrega", "atraso", "péssimo", "recomendo"]:
    if p in vocab:
        print(p, "->", vizinhos(p))

entrega -> [('aentrega', 0.762), ('deentrega', 0.761), ('entregaâ', 0.742), ('entregação', 0.726), ('reentrega', 0.722), ('entregar', 0.712), ('entregares', 0.676), ('entregai', 0.668)]
atraso -> [('atrasos', 0.796), ('oatraso', 0.781), ('00atraso', 0.779), ('atrasou', 0.737), ('000atraso', 0.731), ('atrasado', 0.715), ('atrasamento', 0.708), ('atraso.o', 0.706)]
péssimo -> [('mau', 0.766), ('péssima', 0.754), ('ruim', 0.72), ('péssimos', 0.714), ('ótimo', 0.703), ('bom', 0.694), ('ruim/péssimo', 0.69), ('pessimo', 0.655)]
recomendo -> [('recomendo-me', 0.862), ('recomendo-o', 0.855), ('recomendo-vos', 0.85), ('recomendo-a', 0.842), ('recomendo-te', 0.826), ('recomende', 0.825), ('recomendes', 0.824), ('recomendei', 0.817)]


In [4]:
import re
from collections import Counter
import pandas as pd

df = pd.read_parquet("../data/processed/base_limpa_final.parquet")

def tokenizar(texto):
    return re.findall(r"\w+", texto.lower())

total, dentro = 0, 0
fora = Counter()

for texto in df["texto"]:
    for tok in tokenizar(texto):
        total += 1
        if tok in vocab:
            dentro += 1
        else:
            fora[tok] += 1

print(f"Tokens totais: {total}")
print(f"Cobertura do vocabulário NILC: {dentro / total:.2%}")
print(f"Tokens fora do vocabulário: {total - dentro} ({(total - dentro) / total:.2%})")
print("\n30 tokens fora do vocabulário mais frequentes:")
print(fora.most_common(30))

Tokens totais: 476936
Cobertura do vocabulário NILC: 97.78%
Tokens fora do vocabulário: 10595 (2.22%)

30 tokens fora do vocabulário mais frequentes:
[('2', 916), ('1', 780), ('3', 474), ('10', 407), ('4', 253), ('5', 224), ('02', 216), ('01', 204), ('03', 192), ('12', 185), ('à', 166), ('2018', 163), ('20', 156), ('100', 135), ('04', 135), ('15', 120), ('18', 116), ('17', 114), ('08', 111), ('30', 110), ('11', 108), ('2017', 105), ('05', 100), ('6', 99), ('06', 88), ('07', 80), ('8', 73), ('7', 72), ('às', 66), ('19', 65)]


In [5]:
# tesntando embeddings de comentários
import numpy as np
import pandas as pd

emb = np.load("../data/processed/embeddings.npy")
df = pd.read_parquet("../data/processed/reviews_indexed.parquet")

print(emb.shape, len(df))

def vizinhos_de_comentario(i, k=5):
    sims = emb @ emb[i]  # produto escalar = cosseno, pois os vetores têm norma 1
    melhores = np.argsort(-sims)[1 : k + 1]
    print(f"\nCOMENTÁRIO [{df.loc[i, 'review_score']}★]: {df.loc[i, 'texto']}")
    for j in melhores:
        print(f"  {sims[j]:.3f} [{df.loc[j, 'review_score']}★] {df.loc[j, 'texto']}")

i_atraso = df.index[df["texto"].str.contains("atraso", case=False)][0]
for i in [0, 100, 5000, i_atraso]:
    vizinhos_de_comentario(i)

(40393, 300) 40393

COMENTÁRIO [5★]: Recebi bem antes do prazo estipulado.
  0.981 [5★] Recebi antes do prazo estipulado
  0.977 [4★] Recebi bem antes do prazo previsto.
  0.975 [5★] Recebi bem antes do prazo
  0.975 [5★] Recebi bem antes do prazo.
  0.975 [5★] Recebi bem antes do prazo

COMENTÁRIO [1★]: o meu produto não chegou... segundo o rastreio está parado em Ribeirão Preto sp...
  0.927 [1★] Fiquei acompanhado o produto é ele estálava parado em Ribeirão Preto desdo dia 27 de fevereiro. Só agora, depois da data que está vindo pra São Paulo.
  0.910 [2★] O produto chegou no prazo porém o item veio errado, entrei em contato com o vendedor e o mesmo disse que me mandaria o produto correto. Estou aguardando o código de rastreio do produto correto.
  0.904 [1★] O produto está com a entrega em atraso. Até o momento não obtive nenhum retorno do meu chamado que está em aberto!
  0.903 [4★] Comprei um relógio preto, porém chegou um vermelho, resolvi ficar com esse por também ter combinado